# Chapter 09: Distributions on Spheres

Source span: printed pages 159-192; PDF pages 175-208. The source pages were used for orientation only: concept order, terminology, and the chapter boundary. This notebook uses original prose, synthetic data, generated diagrams, and executable checks.

## Chapter Goal

This chapter moves directional statistics from the circle to the sphere and to the general unit sphere $S^{p-1}=\{x\in\mathbb{R}^p:x^Tx=1\}$. The working question is:

**Which summaries and model families still make geometric sense when observations are points on a sphere or antipodal axes rather than ordinary Euclidean vectors?**

The lesson keeps four objects visible throughout:

- a spherical observation is a unit vector, while an axial observation identifies $x$ and $-x$;
- the mean resultant vector gives a pole when the data are genuinely directional;
- the moment of inertia matrix sees axial and girdle structure even when the mean vector cancels;
- Fisher/von Mises-Fisher, Watson, Bingham, and tangent-limit approximations encode different shapes of probability on a sphere.


## Computational Translation Guide

| Book concept | Computational object used here | What to inspect |
| --- | --- | --- |
| Point on $S^{p-1}$ | Row-normalized NumPy array with norm one | Norm residuals stay below tolerance before any statistic is trusted. |
| Directional sample | Resultant vector $\bar x=n^{-1}\sum_i x_i$, length $R=\|\bar x\|$, mean direction $\hat\mu=\bar x/R$ when $R>0$ | Rotating all observations rotates $\hat\mu$ and leaves $R$ unchanged. |
| Axial sample | Antipodal pairs or sign-invariant densities $f(x)=f(-x)$ | The mean may vanish, but the leading eigenvector of $T=n^{-1}\sum_i x_ix_i^T$ still marks the axis. |
| Moment/inertia matrix | Symmetric positive semidefinite matrix $T=X^TX/n$ with $\operatorname{tr}T=1$ | Eigenvalues describe pole-like, girdle-like, or diffuse structure. |
| Fisher/vMF density | $f(x)\propto \exp(\kappa\mu^Tx)$ on the sphere | Increasing $\kappa$ concentrates mass near one pole. |
| Watson/Bingham axial families | Quadratic densities such as $\exp\{\kappa(\mu^Tx)^2\}$ or $\exp(x^TAx)$ | Sign invariance creates bipolar lobes or equatorial girdles. |
| High-concentration limit | Tangent coordinates scaled by $\sqrt\kappa$ | The curved sphere locally looks like a Gaussian tangent plane. |
| High-dimensional limit | Coordinates of a uniform point scaled by $\sqrt p$ | Any fixed coordinate becomes nearly standard normal as $p$ grows. |

### Chapter Storyboard

1. Build synthetic spherical and axial samples, then compare the mean resultant with the inertia spectrum.
2. Save a Lambert equal-area projection and matrix-summary artifact that makes the directional-versus-axial distinction visible.
3. Plot Fisher/vMF and axial density families on the unit sphere as a rotatable Plotly artifact.
4. Check density positivity and normalization in the $S^2$ case where one-dimensional integration over $t=\mu^Tx$ is transparent.
5. Simulate high-concentration and high-dimensional limits and record the numeric residuals that support the approximations.


In [ ]:
from pathlib import Path
import json
import math
import sys


def find_book_root(start: Path) -> Path:
    start = start.resolve()
    candidates = [start, *start.parents, start / "Directional-Statistics"]
    candidates.extend(parent / "Directional-Statistics" for parent in start.parents)
    seen = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if (
            (candidate / "AGENTS.md").exists()
            and (candidate / "scripts" / "validate_dirstats_course.py").exists()
            and (candidate / "utils").exists()
        ):
            return candidate
    raise RuntimeError("Could not locate the Directional-Statistics course root")


BOOK_ROOT = find_book_root(Path.cwd())
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

TOPIC = "chapter-09"
SOURCE_SPAN = {"printed": "159-192", "pdf": "175-208"}
ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / TOPIC
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

SOURCE_SPAN


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from scipy import integrate, special, stats
from scipy.stats import vonmises_fisher
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from utils.artifacts import display_artifact, save_json, save_matplotlib, save_plotly_html
from utils.validation import assert_artifacts

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(202609)


In [ ]:
def unit_vector(v):
    v = np.asarray(v, dtype=float)
    norm = np.linalg.norm(v)
    if norm == 0:
        raise ValueError("zero vector has no direction")
    return v / norm


def normalize_rows(a):
    a = np.asarray(a, dtype=float)
    norms = np.linalg.norm(a, axis=1, keepdims=True)
    if np.any(norms == 0):
        raise ValueError("zero row cannot be normalized onto a sphere")
    return a / norms


def tangent_frame(mu):
    mu = unit_vector(mu)
    anchor = np.array([1.0, 0.0, 0.0])
    if abs(mu @ anchor) > 0.85:
        anchor = np.array([0.0, 1.0, 0.0])
    e1 = anchor - (anchor @ mu) * mu
    e1 = unit_vector(e1)
    e2 = np.cross(mu, e1)
    return e1, e2, mu


def sample_uniform_sphere(n, p, rng):
    return normalize_rows(rng.normal(size=(n, p)))


def sample_vmf(mu, kappa, n, rng):
    return np.asarray(vonmises_fisher.rvs(mu=unit_vector(mu), kappa=kappa, size=n, random_state=rng))


def sample_antipodal_cluster(axis, kappa, n_pairs, rng):
    half = sample_vmf(axis, kappa, n_pairs, rng)
    sample = np.vstack([half, -half])
    rng.shuffle(sample, axis=0)
    return sample


def summary_stats(x):
    x = np.asarray(x, dtype=float)
    n = x.shape[0]
    mean_vector = x.mean(axis=0)
    R = float(np.linalg.norm(mean_vector))
    mean_direction = mean_vector / R if R > 1e-12 else np.full(x.shape[1], np.nan)
    T = (x.T @ x) / n
    centered = x - mean_vector
    S = (centered.T @ centered) / (n - 1)
    eigvals, eigvecs = np.linalg.eigh(T)
    order = eigvals.argsort()[::-1]
    return {
        "n": n,
        "mean_vector": mean_vector,
        "R": R,
        "mean_direction": mean_direction,
        "T": T,
        "S": S,
        "eigvals": eigvals[order],
        "eigvecs": eigvecs[:, order],
    }


def lambert_equal_area(x, pole=np.array([0.0, 0.0, 1.0])):
    x = np.atleast_2d(np.asarray(x, dtype=float))
    pole = unit_vector(pole)
    e1, e2, pole = tangent_frame(pole)
    t = np.clip(x @ pole, -1.0, 1.0)
    tangent = x - t[:, None] * pole
    tangent_norm = np.linalg.norm(tangent, axis=1)
    r = np.sqrt(2.0 * (1.0 - t))
    uv = np.zeros((len(x), 2))
    good = tangent_norm > 1e-12
    tangent_unit = np.zeros_like(tangent)
    tangent_unit[good] = tangent[good] / tangent_norm[good, None]
    uv[:, 0] = r * (tangent_unit @ e1)
    uv[:, 1] = r * (tangent_unit @ e2)
    return uv


def fisher_density_uniform_t(t, kappa):
    t = np.asarray(t, dtype=float)
    if abs(kappa) < 1e-12:
        return np.ones_like(t)
    return (kappa / np.sinh(kappa)) * np.exp(kappa * t)


def fisher_A3(kappa):
    if abs(kappa) < 1e-8:
        return kappa / 3.0
    return (1.0 / np.tanh(kappa)) - (1.0 / kappa)


def vMF_Ap(kappa, p):
    if abs(kappa) < 1e-8:
        return kappa / p
    nu = p / 2.0 - 1.0
    return float(special.iv(nu + 1.0, kappa) / special.iv(nu, kappa))


def watson_normalizer_s2(kappa):
    value, _ = integrate.quad(lambda z: 0.5 * np.exp(kappa * z * z), -1.0, 1.0, epsabs=1e-11)
    return value


def watson_density_uniform_t(t, kappa):
    return np.exp(kappa * np.asarray(t) ** 2) / watson_normalizer_s2(kappa)


def bingham_intensity(x, A):
    x = np.asarray(x, dtype=float)
    q = np.einsum("...i,ij,...j->...", x, A, x)
    return np.exp(q - np.max(q))


## Spherical Observations, Mean Resultants, and Inertia

The first trap in spherical statistics is treating every cloud with a small Euclidean average as unstructured. A directional cluster has a large mean resultant and a meaningful pole. An axial cluster can have a mean near zero because every direction is paired with its antipode, yet it still has a strong axis. The inertia matrix $T=n^{-1}\sum x_ix_i^T$ detects that second pattern because $xx^T=(-x)(-x)^T$.

The figure below uses synthetic data. Filled markers are on the near hemisphere of the Lambert equal-area projection and open markers are on the far hemisphere. The static artifact is intentionally two-dimensional: it makes the sign ambiguity and the trace-one inertia spectrum readable in a printed or exported notebook.


In [ ]:
true_pole = unit_vector([0.34, -0.22, 0.91])
polar_sample = sample_vmf(true_pole, kappa=18.0, n=96, rng=rng)
axial_sample = sample_antipodal_cluster(true_pole, kappa=28.0, n_pairs=48, rng=rng)

polar_stats = summary_stats(polar_sample)
axial_stats = summary_stats(axial_sample)

sample_checks = {
    "polar_max_unit_norm_error": float(np.max(np.abs(np.linalg.norm(polar_sample, axis=1) - 1.0))),
    "axial_max_unit_norm_error": float(np.max(np.abs(np.linalg.norm(axial_sample, axis=1) - 1.0))),
    "polar_R": polar_stats["R"],
    "axial_R": axial_stats["R"],
    "polar_inertia_eigenvalues": polar_stats["eigvals"].tolist(),
    "axial_inertia_eigenvalues": axial_stats["eigvals"].tolist(),
}
sample_checks


In [ ]:
def draw_lambert_panel(ax, x, stats_dict, title, mode, color):
    uv = lambert_equal_area(x)
    near = x[:, 2] >= 0
    ax.add_patch(Circle((0, 0), 2.0, fill=False, lw=1.2, ec="0.35"))
    ax.axhline(0, color="0.86", lw=0.8)
    ax.axvline(0, color="0.86", lw=0.8)
    ax.scatter(uv[near, 0], uv[near, 1], s=28, c=color, alpha=0.75, edgecolor="white", linewidth=0.35)
    ax.scatter(uv[~near, 0], uv[~near, 1], s=34, facecolors="none", edgecolors=color, alpha=0.9, linewidth=1.0)
    if mode == "directional":
        m_uv = lambert_equal_area(stats_dict["mean_direction"])[0]
        ax.scatter([m_uv[0]], [m_uv[1]], marker="*", s=170, c="#111111", zorder=5, label="mean direction")
        ax.plot([0, m_uv[0]], [0, m_uv[1]], color="#111111", lw=1.6, alpha=0.8)
        label = f"R={stats_dict['R']:.3f}"
    else:
        axis = stats_dict["eigvecs"][:, 0]
        if axis @ true_pole < 0:
            axis = -axis
        axis_uv = lambert_equal_area(np.vstack([axis, -axis]))
        ax.scatter(axis_uv[:, 0], axis_uv[:, 1], marker="D", s=72, c="#111111", zorder=5, label="principal axis")
        ax.plot(axis_uv[:, 0], axis_uv[:, 1], color="#111111", lw=1.3, alpha=0.8)
        label = f"R={stats_dict['R']:.3f}, lambda1={stats_dict['eigvals'][0]:.3f}"
    ax.set_title(title, fontsize=11, loc="left")
    ax.text(-1.92, -1.84, label, fontsize=9, bbox={"boxstyle": "round,pad=0.25", "fc": "white", "ec": "0.75"})
    ax.set_aspect("equal")
    ax.set_xlim(-2.08, 2.08)
    ax.set_ylim(-2.08, 2.08)
    ax.set_xticks([])
    ax.set_yticks([])


fig, axes = plt.subplots(2, 2, figsize=(10.8, 8.8), constrained_layout=True)
draw_lambert_panel(axes[0, 0], polar_sample, polar_stats, "Directional cluster: resultant points to one pole", "directional", "#2563eb")
draw_lambert_panel(axes[0, 1], axial_sample, axial_stats, "Axial cluster: signs cancel but the axis remains", "axial", "#d97706")

xpos = np.arange(3)
width = 0.34
axes[1, 0].bar(xpos - width / 2, polar_stats["eigvals"], width, label="directional", color="#2563eb", alpha=0.82)
axes[1, 0].bar(xpos + width / 2, axial_stats["eigvals"], width, label="axial", color="#d97706", alpha=0.82)
axes[1, 0].set_xticks(xpos, ["lambda1", "lambda2", "lambda3"])
axes[1, 0].set_ylim(0, 1.02)
axes[1, 0].set_ylabel("eigenvalue of T")
axes[1, 0].set_title("Inertia spectra: trace(T)=1", fontsize=11, loc="left")
axes[1, 0].legend(frameon=False)
axes[1, 0].grid(axis="y", color="0.9")

labels = ["directional R", "directional lambda1", "axial R", "axial lambda1"]
values = [polar_stats["R"], polar_stats["eigvals"][0], axial_stats["R"], axial_stats["eigvals"][0]]
colors = ["#2563eb", "#60a5fa", "#d97706", "#fbbf24"]
axes[1, 1].barh(np.arange(len(labels)), values, color=colors)
axes[1, 1].set_yticks(np.arange(len(labels)), labels)
axes[1, 1].set_xlim(0, 1.02)
axes[1, 1].set_xlabel("summary magnitude")
axes[1, 1].set_title("Mean vector versus inertia axis", fontsize=11, loc="left")
for i, v in enumerate(values):
    axes[1, 1].text(min(v + 0.025, 0.94), i, f"{v:.3f}", va="center", fontsize=9)
axes[1, 1].grid(axis="x", color="0.9")

fig.suptitle("Chapter 09: spherical observations need both resultants and inertia", fontsize=14, y=1.02)
mean_inertia_path = save_matplotlib(fig, TOPIC, "core", "spherical-observations-mean-inertia.png", dpi=170)
compat_static_path = save_matplotlib(fig, TOPIC, "core", "concept-diagnostic.png", dpi=170)
plt.close(fig)
display_artifact(mean_inertia_path, width=900)
mean_inertia_path


## Invariant Scaffold: What the Summaries Must Satisfy

The inertia matrix is not an arbitrary covariance matrix. Because every observation has unit norm, $\operatorname{tr}(T)=1$. If $S=(n-1)^{-1}\sum_i(x_i-\bar x)(x_i-\bar x)^T$, then

$$
T = \frac{n-1}{n}S + \bar x\bar x^T,
\qquad
\frac{n-1}{n}\operatorname{tr}(S)+R^2=1.
$$

These identities are useful because they fail immediately if an upstream operation has lost the sphere constraint. The same cell also checks rotational behavior: rotate the data, and the inertia eigenvalues and resultant length should not change.


In [ ]:
def random_rotation_3(rng):
    q, r = np.linalg.qr(rng.normal(size=(3, 3)))
    if np.linalg.det(q) < 0:
        q[:, 0] *= -1
    return q


def inertia_identity_checks(name, x, stats_dict):
    n = stats_dict["n"]
    T = stats_dict["T"]
    S = stats_dict["S"]
    R = stats_dict["R"]
    Q = random_rotation_3(rng)
    rotated = x @ Q.T
    rotated_stats = summary_stats(rotated)
    identity_matrix = ((n - 1) / n) * S + np.outer(stats_dict["mean_vector"], stats_dict["mean_vector"])
    return {
        f"{name}_T_symmetric_error": float(np.max(np.abs(T - T.T))),
        f"{name}_T_trace": float(np.trace(T)),
        f"{name}_T_min_eigenvalue": float(np.min(np.linalg.eigvalsh(T))),
        f"{name}_T_decomposition_error": float(np.max(np.abs(T - identity_matrix))),
        f"{name}_trace_variance_identity_error": float(abs(((n - 1) / n) * np.trace(S) + R * R - 1.0)),
        f"{name}_rotation_R_error": float(abs(rotated_stats["R"] - R)),
        f"{name}_rotation_eigenvalue_error": float(np.max(np.abs(rotated_stats["eigvals"] - stats_dict["eigvals"]))),
    }

summary_invariant_checks = {}
summary_invariant_checks.update(inertia_identity_checks("directional", polar_sample, polar_stats))
summary_invariant_checks.update(inertia_identity_checks("axial", axial_sample, axial_stats))
summary_invariant_checks


## Fisher/von Mises-Fisher and Axial Model Families

On $S^2$, the Fisher distribution is the three-dimensional von Mises-Fisher model. Written as a density with respect to the uniform probability measure on the sphere,

$$
f(x;\mu,\kappa)=\frac{\kappa}{\sinh\kappa}\exp(\kappa\mu^Tx).
$$

It is directional: $x$ and $-x$ do not receive the same density unless $\kappa=0$. Axial families are different. Watson densities use $\exp\{\kappa(\mu^Tx)^2\}$, producing two antipodal poles for $\kappa>0$ and a girdle for $\kappa<0$. Bingham densities use a full quadratic form $\exp(x^TAx)$ and can express unequal spread around an axis or a girdle.

The Plotly artifact below is a model-family gallery rather than a data plot. Rotate it and compare where each density places mass: one pole, two poles, an equator, or an oval axial lobe.


In [ ]:
def sphere_grid(n_theta=72, n_phi=144):
    theta = np.linspace(0.0, np.pi, n_theta)
    phi = np.linspace(0.0, 2.0 * np.pi, n_phi)
    theta_grid, phi_grid = np.meshgrid(theta, phi, indexing="ij")
    x = np.sin(theta_grid) * np.cos(phi_grid)
    y = np.sin(theta_grid) * np.sin(phi_grid)
    z = np.cos(theta_grid)
    xyz = np.stack([x, y, z], axis=-1)
    return x, y, z, xyz


def add_density_surface(fig, row, col, title, color_values, x, y, z, colorscale):
    fig.add_trace(
        go.Surface(
            x=x,
            y=y,
            z=z,
            surfacecolor=color_values,
            colorscale=colorscale,
            showscale=False,
            cmin=float(np.min(color_values)),
            cmax=float(np.max(color_values)),
            opacity=0.98,
            hovertemplate="x=%{x:.2f}<br>y=%{y:.2f}<br>z=%{z:.2f}<br>relative density=%{surfacecolor:.3f}<extra>" + title + "</extra>",
        ),
        row=row,
        col=col,
    )


def make_density_gallery():
    x, y, z, xyz = sphere_grid()
    mu = unit_vector([0.32, -0.18, 0.93])
    t = np.einsum("...i,i->...", xyz, mu)
    A = np.diag([3.8, 0.8, -4.6])
    fisher = fisher_density_uniform_t(t, 6.0)
    watson_poles = watson_density_uniform_t(t, 5.0)
    watson_girdle = watson_density_uniform_t(t, -7.0)
    bingham = bingham_intensity(xyz, A)
    gallery = make_subplots(
        rows=2,
        cols=2,
        specs=[[{"type": "surface"}, {"type": "surface"}], [{"type": "surface"}, {"type": "surface"}]],
        subplot_titles=("Fisher/vMF: one pole", "Watson: antipodal poles", "Watson: girdle", "Bingham: quadratic axial oval"),
        horizontal_spacing=0.02,
        vertical_spacing=0.06,
    )
    add_density_surface(gallery, 1, 1, "Fisher/vMF", fisher, x, y, z, "Viridis")
    add_density_surface(gallery, 1, 2, "Watson poles", watson_poles, x, y, z, "Cividis")
    add_density_surface(gallery, 2, 1, "Watson girdle", watson_girdle, x, y, z, "Plasma")
    add_density_surface(gallery, 2, 2, "Bingham", bingham, x, y, z, "Turbo")
    gallery.update_layout(
        title="Directional and axial densities on S^2",
        height=760,
        margin={"l": 0, "r": 0, "b": 0, "t": 70},
        paper_bgcolor="white",
    )
    for scene_name in ["scene", "scene2", "scene3", "scene4"]:
        gallery.update_layout({
            scene_name: {
                "aspectmode": "data",
                "xaxis": {"visible": False},
                "yaxis": {"visible": False},
                "zaxis": {"visible": False},
                "camera": {"eye": {"x": 1.45, "y": 1.35, "z": 0.9}},
            }
        })
    diagnostics = {
        "fisher_density_min": float(np.min(fisher)),
        "watson_positive_density_min": float(np.min(watson_poles)),
        "watson_girdle_density_min": float(np.min(watson_girdle)),
        "bingham_intensity_min": float(np.min(bingham)),
        "bingham_sign_symmetry_error": float(np.max(np.abs(bingham_intensity(xyz, A) - bingham_intensity(-xyz, A)))),
    }
    return gallery, diagnostics


density_gallery, density_gallery_checks = make_density_gallery()
density_gallery_path = save_plotly_html(
    density_gallery,
    TOPIC,
    "interactive",
    "fisher-watson-bingham-density-gallery.html",
    include_plotlyjs=True,
)
compat_density_gallery_path = save_plotly_html(
    density_gallery,
    TOPIC,
    "interactive",
    "exploration.html",
    include_plotlyjs=True,
)
display_artifact(density_gallery_path, width="100%", height=620)
density_gallery_checks


In [ ]:
def density_normalization_checks():
    fisher_kappa = 6.0
    fisher_integral, fisher_error = integrate.quad(
        lambda z: 0.5 * fisher_density_uniform_t(z, fisher_kappa), -1.0, 1.0, epsabs=1e-11
    )
    fisher_mean_t, _ = integrate.quad(
        lambda z: 0.5 * z * fisher_density_uniform_t(z, fisher_kappa), -1.0, 1.0, epsabs=1e-11
    )
    watson_pos_integral, _ = integrate.quad(
        lambda z: 0.5 * watson_density_uniform_t(z, 5.0), -1.0, 1.0, epsabs=1e-11
    )
    watson_girdle_integral, _ = integrate.quad(
        lambda z: 0.5 * watson_density_uniform_t(z, -7.0), -1.0, 1.0, epsabs=1e-11
    )
    grid = np.linspace(-1, 1, 801)
    return {
        "fisher_s2_integral": float(fisher_integral),
        "fisher_s2_integral_quad_error": float(fisher_error),
        "fisher_A3_numeric_error": float(abs(fisher_mean_t - fisher_A3(fisher_kappa))),
        "fisher_grid_min_density": float(np.min(fisher_density_uniform_t(grid, fisher_kappa))),
        "watson_positive_integral": float(watson_pos_integral),
        "watson_girdle_integral": float(watson_girdle_integral),
        "watson_positive_grid_min_density": float(np.min(watson_density_uniform_t(grid, 5.0))),
        "watson_girdle_grid_min_density": float(np.min(watson_density_uniform_t(grid, -7.0))),
    }


density_checks = density_normalization_checks()
density_checks


## High-Concentration and Asymptotic Intuition

There are two different limits worth keeping apart.

In a high-concentration Fisher/vMF model, most points live close to a mean pole. If $t=\mu^Tx$, then $2\kappa(1-t)$ behaves like a chi-square variable with $p-1$ degrees of freedom; on $S^2$ this has two degrees of freedom. The tangent coordinates, scaled by $\sqrt\kappa$, look approximately Gaussian.

In a high-dimensional uniform sphere, concentration has a different meaning. A fixed coordinate of a uniform point is tiny, but $\sqrt p\,x_1$ is approximately standard normal. This is the geometric reason many high-dimensional sphere calculations turn into Gaussian calculations after the right scaling.


In [ ]:
limit_rng = np.random.default_rng(202610)
limit_mu = np.array([0.0, 0.0, 1.0])
limit_kappa = 35.0
limit_sample = sample_vmf(limit_mu, limit_kappa, n=6000, rng=limit_rng)
e1, e2, _ = tangent_frame(limit_mu)
tangent_coords = np.column_stack([limit_sample @ e1, limit_sample @ e2]) * math.sqrt(limit_kappa)
t = limit_sample @ limit_mu
radial_chisq = 2.0 * limit_kappa * (1.0 - t)
tangent_cov = np.cov(tangent_coords.T)

p_values = [6, 50, 200]
high_dim_samples = {p: sample_uniform_sphere(6000, p, limit_rng)[:, 0] * math.sqrt(p) for p in p_values}

uniform_null_p = 3
uniform_null_n = 160
uniform_null_reps = 3500
null_stats = []
for _ in range(uniform_null_reps):
    sample = sample_uniform_sphere(uniform_null_n, uniform_null_p, limit_rng)
    R = np.linalg.norm(sample.mean(axis=0))
    null_stats.append(uniform_null_n * uniform_null_p * R * R)
null_stats = np.asarray(null_stats)

fig, axes = plt.subplots(2, 2, figsize=(11.0, 8.2), constrained_layout=True)

axes[0, 0].scatter(tangent_coords[:900, 0], tangent_coords[:900, 1], s=8, alpha=0.25, color="#2563eb")
for radius in [1.0, 2.0, 3.0]:
    axes[0, 0].add_patch(Circle((0, 0), radius, fill=False, ec="0.75", lw=0.8))
axes[0, 0].set_aspect("equal")
axes[0, 0].set_xlim(-4.2, 4.2)
axes[0, 0].set_ylim(-4.2, 4.2)
axes[0, 0].set_title("sqrt(kappa) tangent coordinates", fontsize=11, loc="left")
axes[0, 0].set_xlabel("tangent coordinate 1")
axes[0, 0].set_ylabel("tangent coordinate 2")
axes[0, 0].grid(color="0.92")

xgrid = np.linspace(0, 12, 300)
axes[0, 1].hist(radial_chisq, bins=50, density=True, color="#60a5fa", alpha=0.72, label="simulation")
axes[0, 1].plot(xgrid, stats.chi2.pdf(xgrid, df=2), color="#111111", lw=1.8, label="chi-square df=2")
axes[0, 1].set_title("2*kappa*(1-mu'x) high-concentration limit", fontsize=11, loc="left")
axes[0, 1].set_xlabel("scaled radial loss")
axes[0, 1].legend(frameon=False)
axes[0, 1].grid(axis="y", color="0.92")

normal_grid = np.linspace(-4, 4, 300)
for p, values in high_dim_samples.items():
    axes[1, 0].hist(values, bins=70, density=True, histtype="step", lw=1.35, label=f"p={p}")
axes[1, 0].plot(normal_grid, stats.norm.pdf(normal_grid), color="#111111", lw=1.9, label="N(0,1)")
axes[1, 0].set_title("sqrt(p) times one uniform-sphere coordinate", fontsize=11, loc="left")
axes[1, 0].set_xlabel("scaled coordinate")
axes[1, 0].legend(frameon=False)
axes[1, 0].grid(axis="y", color="0.92")

xnull = np.linspace(0, 16, 320)
axes[1, 1].hist(null_stats, bins=55, density=True, color="#d97706", alpha=0.65, label="uniform samples")
axes[1, 1].plot(xnull, stats.chi2.pdf(xnull, df=uniform_null_p), color="#111111", lw=1.9, label="chi-square df=3")
axes[1, 1].set_title("n*p*R^2 under uniformity", fontsize=11, loc="left")
axes[1, 1].set_xlabel("Rayleigh scaling")
axes[1, 1].legend(frameon=False)
axes[1, 1].grid(axis="y", color="0.92")

fig.suptitle("Chapter 09: curved-sphere limits become tangent or Gaussian after scaling", fontsize=14, y=1.02)
limit_path = save_matplotlib(fig, TOPIC, "core", "tangent-and-high-dimensional-limits.png", dpi=170)
plt.close(fig)
display_artifact(limit_path, width=900)

limit_checks = {
    "tangent_scaled_mean_norm": float(np.linalg.norm(tangent_coords.mean(axis=0))),
    "tangent_scaled_covariance": tangent_cov.tolist(),
    "tangent_scaled_covariance_identity_error": float(np.max(np.abs(tangent_cov - np.eye(2)))),
    "radial_chisq_mean_error_df2": float(abs(radial_chisq.mean() - 2.0)),
    "radial_chisq_var_error_df2": float(abs(radial_chisq.var() - 4.0)),
    "high_dim_coordinate_variance": {str(p): float(np.var(values)) for p, values in high_dim_samples.items()},
    "uniform_null_chisq_mean_error_df3": float(abs(null_stats.mean() - uniform_null_p)),
    "uniform_null_chisq_var_error_df3": float(abs(null_stats.var() - 2 * uniform_null_p)),
}
limit_checks


## Applied Lab: Choose the Statistic That Matches the Scientific Object

Use this cell as a small design exercise. The same physical measurements can be recorded as directions or axes. If signs matter, summarize the sample by $\bar x$ and a Fisher/vMF model may be appropriate. If signs do not matter, use the inertia matrix and an axial family.

The printed output makes the decision visible: `R` is large for the directional sample and nearly zero for the antipodal sample, while the leading inertia eigenvalue stays large in both cases. That is the computational signature of the difference between a pole and an axis.


In [ ]:
def model_decision_row(name, stats_dict):
    leading_axis = stats_dict["eigvecs"][:, 0]
    if leading_axis @ true_pole < 0:
        leading_axis = -leading_axis
    return {
        "sample": name,
        "R_mean_resultant": round(float(stats_dict["R"]), 4),
        "lambda1_inertia": round(float(stats_dict["eigvals"][0]), 4),
        "lambda2_inertia": round(float(stats_dict["eigvals"][1]), 4),
        "lambda3_inertia": round(float(stats_dict["eigvals"][2]), 4),
        "recommended_summary": "mean direction + vMF/Fisher" if stats_dict["R"] > 0.35 else "principal axis + Watson/Bingham",
        "principal_axis_alignment_with_true_axis": round(float(abs(leading_axis @ true_pole)), 4),
    }

lab_decisions = [model_decision_row("directional synthetic sample", polar_stats), model_decision_row("antipodal axial sample", axial_stats)]
lab_decisions


## Final Sanity Checks

The final cell records checks that audit both geometry and artifacts:

- all generated observations used in the lesson remain on their intended unit spheres;
- Fisher and Watson densities on $S^2$ are positive and normalize numerically with respect to the uniform sphere measure;
- inertia matrices are symmetric, positive semidefinite, trace one, and obey the variance decomposition;
- rotations preserve resultant lengths and inertia spectra;
- concept-named artifacts exist under `artifacts/chapter-09` and have nonzero size.


In [ ]:
# Display saved compatibility/check artifacts so static exports show every visual save target.
display_artifact(mean_inertia_path, width=900)
display_artifact(density_gallery_path, width="100%", height=760)
display_artifact(limit_path, width=900)


In [ ]:
numeric_checks = {
    "source_span": SOURCE_SPAN,
    "unit_norms": sample_checks,
    "inertia_identities": summary_invariant_checks,
    "density_normalization_and_positivity": {**density_checks, **density_gallery_checks},
    "asymptotic_limit_diagnostics": limit_checks,
    "lab_decisions": lab_decisions,
}

boolean_checks = {
    "polar_vectors_unit_norm": sample_checks["polar_max_unit_norm_error"] < 1e-12,
    "axial_vectors_unit_norm": sample_checks["axial_max_unit_norm_error"] < 1e-12,
    "directional_R_in_unit_interval": 0.0 <= polar_stats["R"] <= 1.0,
    "axial_R_in_unit_interval": 0.0 <= axial_stats["R"] <= 1.0,
    "directional_inertia_psd": summary_invariant_checks["directional_T_min_eigenvalue"] > -1e-12,
    "axial_inertia_psd": summary_invariant_checks["axial_T_min_eigenvalue"] > -1e-12,
    "directional_inertia_trace_one": abs(summary_invariant_checks["directional_T_trace"] - 1.0) < 1e-12,
    "axial_inertia_trace_one": abs(summary_invariant_checks["axial_T_trace"] - 1.0) < 1e-12,
    "inertia_decomposition_identity": max(
        summary_invariant_checks["directional_T_decomposition_error"],
        summary_invariant_checks["axial_T_decomposition_error"],
    ) < 1e-12,
    "fisher_density_normalizes": abs(density_checks["fisher_s2_integral"] - 1.0) < 1e-10,
    "watson_densities_normalize": max(
        abs(density_checks["watson_positive_integral"] - 1.0),
        abs(density_checks["watson_girdle_integral"] - 1.0),
    ) < 1e-10,
    "densities_positive_on_grid": min(
        density_checks["fisher_grid_min_density"],
        density_checks["watson_positive_grid_min_density"],
        density_checks["watson_girdle_grid_min_density"],
        density_gallery_checks["bingham_intensity_min"],
    ) > 0.0,
    "bingham_is_antipodally_symmetric": density_gallery_checks["bingham_sign_symmetry_error"] < 1e-12,
    "tangent_covariance_reasonable": limit_checks["tangent_scaled_covariance_identity_error"] < 0.16,
    "uniform_null_scaling_reasonable": limit_checks["uniform_null_chisq_mean_error_df3"] < 0.16,
}

failed = {name: value for name, value in boolean_checks.items() if not value}
assert not failed, failed

check_payload = {"numeric": numeric_checks, "booleans": boolean_checks}
checks_path = save_json(check_payload, TOPIC, "checks", "chapter-09-numeric-invariants.json")
compat_checks_path = save_json(check_payload, TOPIC, "checks", "numeric-checks.json")
artifact_records = assert_artifacts(
    [mean_inertia_path, compat_static_path, density_gallery_path, compat_density_gallery_path, limit_path, checks_path, compat_checks_path],
    min_bytes=500,
)
final_sanity = {
    "source_span": SOURCE_SPAN,
    "artifacts": artifact_records,
    "booleans": boolean_checks,
    "pdf_used_for": "orientation only; no prose, tables, figures, screenshots, or page crops copied",
}
final_path = save_json(final_sanity, TOPIC, "checks", "chapter-09-final-sanity.json")
compat_final_path = save_json(final_sanity, TOPIC, "checks", "final-sanity.json")
assert final_path.exists() and final_path.stat().st_size > 500
assert compat_final_path.exists() and compat_final_path.stat().st_size > 500
final_sanity


## Standalone Reading Notes

This chapter is the bridge from circular thinking to genuinely spherical thinking. A point on `S^2` has no privileged longitude-latitude coordinate system; rotations should not change the statistical conclusion. That is why the notebook keeps returning to unit-vector checks, mean resultant vectors, inertia matrices, and axial sign symmetry. These are the objects that survive coordinate changes.

Read the Fisher, Watson, and Bingham views as a model-selection gallery. Fisher models prefer an oriented pole. Watson models are axial and therefore treat antipodal directions consistently. Bingham-style diagnostics use scatter eigenstructure to detect more than one axis of spread. If a later inference problem is unclear, identify whether the data have a preferred direction, a preferred axis, a girdle, or a more general anisotropic pattern before choosing a formula.


## Takeaways

- A spherical observation is constrained before it is summarized. Unit-norm checks are not bookkeeping; they are the sample-space definition.
- The mean resultant is a directional statistic. It is powerful for a one-pole sample and deliberately weak for antipodal axes.
- The inertia matrix is the bridge from data display to axial model families. Its trace, positive semidefiniteness, and eigenvectors are geometric invariants.
- Fisher/von Mises-Fisher densities model a pole. Watson and Bingham densities model sign-invariant axes, girdles, and quadratic spread.
- High concentration turns the sphere into a tangent Gaussian problem. High dimension turns fixed coordinates of uniform-sphere points into Gaussian limits after $\sqrt p$ scaling.
